## Guide, Step 3: Change data and watch the events

### Predictions

| Question | Prediction | Reasoning |
|---|---|---|
| Does an event appear after the INSERT? | Yes, within about a second | Debezium reads the committed change from the WAL. |
| `op`, `before`, `after` | `op` = `c`, `before` = null, `after` = the new row | A new row has no previous state. |
| Which column is the key? | `id` (the primary key) | Debezium uses the table's primary key as the message key. |
| UPDATE and DELETE | `op` = `u` and `d`, plus a tombstone after the delete | General Debezium behaviour. |

### Observation

Connector status (`curl.exe -s http://localhost:8083/connectors/activity-connector/status`):

```
{"name":"activity-connector","connector":{"state":"RUNNING","worker_id":"172.22.0.4:8083","version":"3.3.2.Final"},"tasks":[{"id":0,"state":"RUNNING","worker_id":"172.22.0.4:8083","version":"3.3.2.Final"}],"type":"source"}
```

Commands in `psql` (database `activity`):

```
INSERT INTO activity(id, name) VALUES (1, 'Alice');
UPDATE activity SET email = 'alice@example.com' WHERE id = 1;
DELETE FROM activity WHERE id = 1;
```

Reading the topic:

`docker exec -it kafka kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic dbserver1.public.activity --from-beginning --property print.key=true`

Event 1, INSERT (key, then value; `schema` parts shortened):

```
{"schema":{...},"payload":{"id":1}}    {"schema":{...},"payload":{"before":null,"after":{"id":1,"name":"Alice","email":null},"source":{"version":"3.3.2.Final","connector":"postgresql","name":"dbserver1","ts_ms":1795000593555,"snapshot":"false","db":"activity","sequence":"[null,\"33706272\"]","schema":"public","table":"activity","txId":759,"lsn":33706272,"xmin":null},"transaction":null,"op":"c","ts_ms":1795000594126}}
```

Event 2, UPDATE:

```
{"schema":{...},"payload":{"id":1}}    {"schema":{...},"payload":{"before":null,"after":{"id":1,"name":"Alice","email":"alice@example.com"},"source":{...,"txId":760,"lsn":33706608,"xmin":null},"transaction":null,"op":"u","ts_ms":1795000729757}}
```

Event 3, DELETE:

```
{"schema":{...},"payload":{"id":1}}    {"schema":{...},"payload":{"before":{"id":1,"name":"","email":null},"after":null,"source":{...,"txId":761,"lsn":33706808,"xmin":null},"transaction":null,"op":"d","ts_ms":1795000739474}}
```

Event 4, tombstone:

```
{"schema":{...},"payload":{"id":1}}    null
```

### Analysis

- **The key is `id`.** Nobody configured it: Debezium takes the table's primary key. All events of row 1 share the key `{"id":1}`, so they go to one partition and keep their order.
- **`op` values:** `c` (create), `u` (update), `d` (delete). INSERT has `before: null`, DELETE has `after: null`.
- **`before` on UPDATE is null,** and on DELETE it only holds the key (`name` shows as an empty string). My explanation: PostgreSQL's default replica identity logs only the old key, not the old row values. I did not verify the table's replica identity.
- **Tombstone:** after the delete event there is an event with the same key and value `null`. It lets Kafka's log compaction later remove the history of that key.
- **`lsn` rises** with every change (33706272 → 33706608 → 33706808): it is the position in the write-ahead log, i.e. the commit order. Debezium never queries the table, so it cannot skip rows like the `id > last_id` poller in Exercise 5.

**Key idea:** CDC turns every committed INSERT, UPDATE and DELETE into a Kafka event in commit order, keyed by the primary key.

## Guide, Step 4: Updates, deletes and tombstones (optional)

The UPDATE and DELETE were already run in Step 3, so the outputs are the same (see there).

### Observation

| Command | `op` | `before` | `after` |
|---|---|---|---|
| `INSERT` | `c` | `null` | `{"id":1,"name":"Alice","email":null}` |
| `UPDATE` | `u` | `null` | `{"id":1,"name":"Alice","email":"alice@example.com"}` |
| `DELETE` | `d` | `{"id":1,"name":"","email":null}` (only the key is real) | `null` |
| tombstone | none | none | value `null`, key `{"id":1}` |

### Question 1: Who needs the full old row?

`before` is (almost) empty because PostgreSQL's default `REPLICA IDENTITY` writes only the primary key of the old row into the write-ahead log, and Debezium can only forward what is in the log. `ALTER TABLE ... REPLICA IDENTITY FULL` would log the whole old row.

Consumers that need the full old row: an **audit service** (what changed from what to what), a **fraud model** that compares the new state with the previous one, and an **index keyed by a value** (to find and remove the old entry).

### Question 2: What does a copy of the table do with the tombstone?

A cache, search index or vector index **deletes the entry with that key** (and must handle a `null` value without crashing). Otherwise deleted rows would stay in the copy and show up in search results. For Kafka itself, the tombstone allows **log compaction** to eventually remove all events of that key.

## Exercise 6, Part 1: The decision

### Comparison with the requirements

| Option | R1: payment path untouched | R2: every payment, every model |
|---|---|---|
| 1. Polling | Violated: every agent adds queries on PostgreSQL | Violated: `id > last_id` misses rows that commit out of id order (Exercise 5) |
| 2. Dual write | Violated: the certified payment code must change | Violated: a crash between commit and send loses an event, a send before the commit invents a rolled-back payment |
| 3. Outbox | Violated: the payment code must also write the outbox row | Met: payment and outbox row are in one transaction |
| 4. Log-based CDC | Met: no code change, no queries | Met: every committed change, in commit order |

### Decision

I would choose log-based CDC (option 4): Debezium reads PostgreSQL's write-ahead log and publishes every committed payment to Kafka in commit order. It is the only option that keeps the certified payment path unchanged without adding any query load (R1) and still delivers every committed payment (R2). Polling fails R1 and R2, because every agent adds queries to the system of record and `WHERE id > last_id` misses rows that commit out of order, as we saw in Exercise 5. Dual write needs a change to the payment code and can lose an event after a crash or invent a payment that was rolled back. The cost of CDC is the replication slot: PostgreSQL keeps all WAL that the connector has not read, so if the connector stops for a weekend, the disk can fill and the database, and with it the payment path, would stop. In production I would monitor the WAL retained by the slot, the connector and task status, and the consumer lag, and alert before the disk is at risk. If my team owned the payment path, I would choose the outbox pattern, writing an outbox row in the same transaction as the payment, because it is just as correct and avoids the slot risk.

## Exercise 6, Part 1: The decision (how do the payments reach the models?)

### Scenario

A card processor commits every payment into PostgreSQL, its system of record. The payment path (`card_processor.py`) belongs to another team and is certified, so it must not be changed. Two stateful fraud models (`anomaly` and `velocity`) must see every payment of a cardholder, and case management wants alerts within seconds. The lab runs 1,000 payments/s for 100,000 cardholders.

### Requirements

| | Requirement |
|---|---|
| R1 | System of record untouched: the payment path stays unchanged, never waits for a fraud check, and gets no extra query load. |
| R2 | Every payment to every model: none is missed, and adding another consumer must not change what the others get. |
| R3 | Correct at any scale: each model catches at least 90% of its frauds with few false alerts, however many instances run. |
| R4 | Fresh: an alert at most 5 s (p95) after the payment was committed. |

### Comparison of the four options

| Option | How it works | R1: payment path untouched | R2: every payment, every model | Cost to the database |
|---|---|---|---|---|
| 1. Polling | The agents query the table again and again (`WHERE id > last_id`). | Violated: every agent adds queries to the system of record. | Violated: payments that commit out of `id` order are missed (as in Exercise 5). | Query load for every reader. |
| 2. Dual write | The card processor also sends every payment to Kafka after its commit. | Violated: the certified payment code must change. | Violated: a crash between commit and send loses the event, a send before the commit invents a payment that was rolled back. | None, but the application changes. |
| 3. Outbox | The payment and an outbox row are written in the same transaction, and a relay publishes the outbox to Kafka. | Violated: the payment code must also write the outbox row. | Met: both rows are in one transaction. | One extra row per transaction. |
| 4. Log-based CDC | Debezium reads the write-ahead log (WAL) and publishes every committed change to Kafka. | Met: no code change, no queries on the table. | Met: every commit, in commit order. | A replication slot: PostgreSQL keeps all WAL the connector has not read yet. |

### Decision

I would choose log-based CDC (option 4): Debezium reads PostgreSQL's write-ahead log and publishes every committed payment to Kafka in commit order. It is the only option that keeps the certified payment path unchanged without adding any query load (R1) and still delivers every committed payment (R2). Polling fails R1 and R2, because every agent adds queries to the system of record and `WHERE id > last_id` misses rows that commit out of order, as we saw in Exercise 5. Dual write needs a change to the payment code and can lose an event after a crash or invent a payment that was rolled back. The cost of CDC is the replication slot: PostgreSQL keeps all WAL that the connector has not read, so if the connector stops for a weekend, the disk can fill and the database, and with it the payment path, would stop. In production I would monitor the WAL retained by the replication slot, the connector and task status, and the consumer lag, and alert before the disk is at risk. If my team owned the payment path, I would choose the outbox pattern, writing an outbox row in the same transaction as the payment, because it is just as correct and avoids the slot risk.

### Answers to the four questions

1. **Which option, and why?** Log-based CDC: it is the only option that satisfies R1 and R2 together.
2. **What does it cost?** A replication slot. PostgreSQL keeps all WAL that the connector has not read, so a long connector outage can fill the disk and stop the database and the payment path.
3. **How would I notice before it hurts?** Monitor the WAL retained by the slot (`pg_replication_slots` and disk usage), the connector and task status (`RUNNING`), and the consumer lag. Alert on growth, and have a procedure for dropping the slot if the connector cannot be revived. *(These monitoring points are my own suggestions, based on how replication slots work.)*
4. **If my team owned the payment path?** The outbox pattern: it is correct and complete, avoids the replication slot risk, and its cost is one extra row per transaction and a change to the application, which is acceptable when you own it.

### Related observation from the Guide

In the Guide, Debezium turned every committed INSERT, UPDATE and DELETE into a Kafka event in commit order, without querying the table, and chose the primary key as the message key.

## Exercise 6, Part 2, Run 1: Debezium's defaults

### Prediction

The three velocity instances do **not** share the work, and R4 fails. `fraud-connector.json` sets no partitions or key, so the topic is created with the broker default (expected: 1 partition) and the key is the table's primary key (transaction id). With 1 partition, only one velocity instance gets work and the others stay idle. One instance handles at most 400 to 500 payments/s, but 1,000/s arrive, so the lag grows and alerts come far later than 5 s.

### Observation

Commands (terminal 1): `python connector.py fraud-connector.json`, then `python card_processor.py`. Terminal 2: `python agents.py`. Terminal 3 after about 150 s: `python scoreboard.py`.

Agents (start and a few later lines):

```
started 1 anomaly and 3 velocity agent(s). Ctrl+C stops all.
[ano-1] assigned partitions [0]
[vel-3] assigned partitions []
[vel-1] assigned partitions [0]
[vel-2] assigned partitions []
[vel-1]       393 transactions      0 alerts      393 cardholders  lag   25.2 s  out-of-order 0
[vel-1]     5,763 transactions      0 alerts    5,604 cardholders  lag   52.0 s  out-of-order 39
[vel-1]    11,962 transactions      0 alerts   11,230 cardholders  lag   71.4 s  out-of-order 41
[vel-1]    19,321 transactions      0 alerts   17,538 cardholders  lag   88.4 s  out-of-order 41
[ano-1]    34,219 transactions      2 alerts   28,941 cardholders  lag   69.3 s  out-of-order 0
```

Card processor (last lines): `12:34:36  89,885 transactions  665/s`

Kafka UI: topic `fraud.public.transactions` has **1 partition**, 81,355 messages, 234 MB (at the time of the screenshot).

Scoreboard:

```
Run 1795001541: started 12:32:21, 99,997 transactions in 150 s (667/s), 16 frauds so far (still running)

Consumer groups on the CDC topic(s)   [R2]
  group                    topic                              partitions  members       lag
  anomaly-agent            fraud.public.transactions                   1        1     5,957
  velocity-agent           fraud.public.transactions                   1        3    23,642

Agent instances (telemetry of this run)   [R3, R4]
  instance         pid model      transactions  cardholders  out-of-order  lag (s)      seen
  ano-1          22852 anomaly          44,417       35,860             0     82.8       6s ago
  vel-1          15456 velocity         24,182       21,462            41    109.0       7s ago

  velocity model (card testing)
    R3  frauds caught          1/7  (14%)   FAIL  (>= 90 %)
    R3  false alerts           0   PASS  (at most 10 % of the flagged transactions)
    R4  alert latency p50/p95  107.7 s / 108.9 s   FAIL  (p95 <= 5 s after the commit)

  anomaly model (account takeover)
    R3  frauds caught          3/8  (38%)   FAIL  (>= 90 %)
    R3  false alerts           0   PASS  (at most 10 % of the flagged transactions)
    R4  alert latency p50/p95  68.2 s / 82.6 s   FAIL  (p95 <= 5 s after the commit)
```

Key of an event (Kafka UI): *(fill in after checking, expected: the transaction id)*

Key of an event (Kafka UI, *Topics → fraud.public.transactions → Messages*, expanded row):

```
"payload": {"id": 177075}
```

The key is only the transaction `id` (the table's primary key). Nobody configured it: Debezium takes it from the table definition. It does not contain the cardholder. All messages show partition 0.

### Explanation

*(Rewrite in your own words.)* Debezium's defaults create the topic with **1 partition**. In a consumer group one partition goes to one consumer, so only `vel-1` works and `vel-2` and `vel-3` are idle (`assigned partitions []`). One velocity instance cannot keep up with the payment rate, so the lag grows (25 s to 109 s) and R4 fails (p95 108.9 s). R3 also fails (1/7 and 3/8 frauds), most likely because the agents were far behind when the scoreboard ran, so alerts had not been produced yet (my interpretation). Partitions and the key are a capacity decision that the defaults do not make for us.

The key is the transaction id, not the cardholder. With one partition this does not matter yet, but with several partitions one cardholder's payments would be spread over different instances, and the stateful models would only see part of each cardholder's history.

## Exercise 6, Part 2, Run 2: more partitions

### Change in `fraud-connector.json`

```json
    "topic.creation.default.partitions": "6",
    "topic.creation.default.replication.factor": "1",
```

(Added inside `config`. Replication factor 1 because there is a single broker.)

### Prediction

All three velocity instances now work (2 partitions each) and the lag drops, so R4 should pass. But **R3 of the velocity model still fails**: the key is the transaction id, not the cardholder, so Kafka spreads one cardholder's payments over different partitions and instances. The velocity model counts a cardholder's payments in the last 60 s and flags the 8th, but each instance only sees a part of them, so many card-testing frauds are not detected.

### Observation

Routine: `Ctrl+C` on agents and card processor, `python connector.py fraud-connector.json`, `python card_processor.py`, `python agents.py`, scoreboard.

#### 1. Connector (old connector, slot and topic removed, new topic created)

```
removing what writes to fraud.* ...
  deleted connector fraud-connector-123139
  dropped replication slot fraudslot
  deleted topic fraud.public.transactions
registered fraud-connector-124840
connector fraud-connector-124840: RUNNING
  table public.transactions -> topic fraud.public.transactions   (appears with the first event)
```

#### 2. Agents

```
started 1 anomaly and 3 velocity agent(s). Ctrl+C stops all.
[vel-2] model velocity, group velocity-agent: fraud.public.transactions -> fraud-alerts
[ano-1] model anomaly, group anomaly-agent: fraud.public.transactions -> fraud-alerts
[vel-1] model velocity, group velocity-agent: fraud.public.transactions -> fraud-alerts
[vel-3] model velocity, group velocity-agent: fraud.public.transactions -> fraud-alerts
[ano-1] assigned partitions [0, 1, 2, 3, 4, 5]
[vel-1] assigned partitions [0, 1]
[vel-2] assigned partitions [2, 3]
[vel-3] assigned partitions [4, 5]
[vel-2]       410 transactions      0 alerts      410 cardholders  lag   15.3 s  out-of-order 0
[vel-1]       146 transactions      0 alerts      146 cardholders  lag   15.2 s  out-of-order 0
[ano-1]       878 transactions      0 alerts      873 cardholders  lag   15.9 s  out-of-order 0
[vel-3]       137 transactions      0 alerts      137 cardholders  lag   15.3 s  out-of-order 0
...
[vel-2]    12,006 transactions      0 alerts   11,327 cardholders  lag   20.4 s  out-of-order 12
[ano-1]    29,000 transactions      1 alerts   25,217 cardholders  lag   28.6 s  out-of-order 0
...
[vel-1]    33,714 transactions      0 alerts   28,629 cardholders  lag   59.4 s  out-of-order 12
[vel-3]    33,764 transactions      0 alerts   28,611 cardholders  lag   57.3 s  out-of-order 15
[ano-1]    82,387 transactions      3 alerts   56,351 cardholders  lag   80.9 s  out-of-order 0
```

Each velocity instance got two partitions, and the anomaly agent got all six.

#### 3. Card processor

```
run 1795002530: 1,000 transactions/s, a fraud every 8 s (first after 30 s). Ctrl+C to stop.
12:48:55      3,600 transactions     715/s
12:49:00      6,300 transactions     628/s
...
12:50:30     96,256 transactions     962/s
...
12:51:15    137,193 transactions     946/s
```

#### 4. Kafka UI

- `fraud.public.transactions` has **6 partitions**, replication factor 1, 193,913 messages at the time of the screenshot, spread evenly (32,061 to 32,560 messages per partition).
- The key of an event is only the transaction id (primary key). The cardholder `user_id` is an ordinary field in the value:

```
"after": {"id": 914139, "user_id": 146135, "amount": 26.18, "card_type": "AMEX", "merchant_id": 4783, "created_at": 1795002766858697}
```

#### 5. Scoreboard after about 130 s (run still going)

```
Run 1795002530: started 12:48:50, 123,281 transactions in 130 s (948/s), 13 frauds so far (still running)

  group                    topic                              partitions  members       lag
  anomaly-agent            fraud.public.transactions                   6        1    53,942
  velocity-agent           fraud.public.transactions                   6        3    40,196

  instance         pid model      transactions  cardholders  out-of-order  lag (s)      seen
  ano-1          12744 anomaly          69,283       50,157             0     53.4       6s ago
  vel-1          11140 velocity         27,247       23,861            11     45.8       6s ago
  vel-2          19720 velocity         28,174       24,586            20     42.3       6s ago
  vel-3          22944 velocity         27,276       23,851            15     45.0       6s ago

  velocity model (card testing)
    R3  frauds caught          0/6  (0%)   FAIL  (>= 90 %)
    R3  false alerts           0   PASS  (at most 10 % of the flagged transactions)
    R4  no alerts; worst lag of the velocity agents 45.8 s   FAIL  (<= 5 s)

  anomaly model (account takeover)
    R3  frauds caught          3/6  (50%)   FAIL  (>= 90 %)
    R3  false alerts           0   PASS  (at most 10 % of the flagged transactions)
    R4  alert latency p50/p95  31.2 s / 48.1 s   FAIL  (p95 <= 5 s after the commit)
```

#### 6. Scoreboard after about 345 s (processor still running, lag kept growing)

```
Run 1795002530: started 12:48:50, 314,755 transactions in 345 s (912/s), 40 frauds so far (still running)

  anomaly-agent            fraud.public.transactions                   6        1   179,988
  velocity-agent           fraud.public.transactions                   6        3   134,237

  velocity model (card testing)
    R3  frauds caught          0/20  (0%)   FAIL  (>= 90 %)
    R4  alert latency p50/p95  78.8 s / 78.8 s   FAIL  (p95 <= 5 s after the commit)

  anomaly model (account takeover)
    R3  frauds caught          7/20  (35%)   FAIL  (>= 90 %)
    R4  alert latency p50/p95  81.5 s / 186.8 s   FAIL  (p95 <= 5 s after the commit)
```

#### 7. Scoreboard after about 378 s (the agents had stopped, `members 0`, so the lag stayed frozen)

```
Run 1795002530: started 12:48:50, 346,382 transactions in 378 s (917/s), 44 frauds so far

  group                    topic                              partitions  members       lag
  anomaly-agent            fraud.public.transactions                   6        0   198,259
  velocity-agent           fraud.public.transactions                   6        0   147,631

  instance         pid model      transactions  cardholders  out-of-order  lag (s)      seen
  ano-1          12744 anomaly         147,693       77,258             0    223.9      27s ago
  vel-1          11140 velocity         65,666       48,215            24    166.1      26s ago
  vel-2          19720 velocity         66,261       48,511            37    165.4      27s ago
  vel-3          22944 velocity         65,943       48,326            29    166.9      26s ago

  velocity model (card testing)
    R3  frauds caught          0/22  (0%)   FAIL  (>= 90 %)
    R3  false alerts           0   PASS  (at most 10 % of the flagged transactions)
    R4  alert latency p50/p95  78.8 s / 78.8 s   FAIL  (p95 <= 5 s after the commit)

  anomaly model (account takeover)
    R3  frauds caught          8/22  (36%)   FAIL  (>= 90 %)
    R3  false alerts           0   PASS  (at most 10 % of the flagged transactions)
    R4  alert latency p50/p95  96.2 s / 214.7 s   FAIL  (p95 <= 5 s after the commit)
```

### Explanation

Every velocity instance now gets two partitions (`[0, 1]`, `[2, 3]`, `[4, 5]`), so no instance is idle any more, unlike in Run 1. But the velocity model still catches almost no frauds: **0 of 22 (0%)**, so R3 fails. The reason is the key. The key of every event is the table's primary key, `{"id": ...}`. The partition is `hash(key) mod 6`, so the 12 test payments of one stolen card land on random partitions and are spread over the three instances. Each instance only sees about 4 of them in its 60 s window, below the threshold of 8, so **no instance sees the whole attack**. The partitions are evenly filled (about 32,000 messages each), which also fits a key that has nothing to do with the cardholder.

The anomaly model should still pass, according to the assignment, because it runs a single instance that reads all six partitions and sees every payment. With two instances it would fail in the same way.

**What differed from the assignment in my run:** R4 failed for both models. Each agent processed only about 175 to 210 payments/s (one velocity instance is expected to handle 400 to 500), and the lag grew from about 15 s to over 160 s. The anomaly model also failed R3 (8/22). My interpretation is that my computer was overloaded: PostgreSQL, Kafka, Connect, the Kafka UI, the card processor and four agents all ran on one machine. I did not verify this, for example with Task Manager. The velocity result (0 caught) is consistent with the key problem, although the backlog could also contribute, since the scoreboard counted only the frauds the agents had already scored.

**The rule:** the key must be the entity whose state the model keeps, here the cardholder (`user_id`), not the primary key of the table. Partitions cap parallelism (Run 1), and the key decides which payments meet in the same instance (Run 2).

## Exercise 6, Part 2, Run 3: the cardholder as the key

### Change in `fraud-connector.json`

```json
    "message.key.columns": "public.transactions:user_id",
```

(Added inside `config`, in addition to the two lines from Run 2. The card processor was started with `--tps 500` to reduce the load on my laptop.)

### Prediction

The key of an event in the Kafka UI becomes `{"user_id": ...}` instead of `{"id": ...}`. Kafka hashes the key to choose the partition, so all payments of one cardholder land in the same partition and are read by the same instance, in commit order. The velocity instance that owns that partition sees all 12 test payments of a stolen card within 60 s and flags the 8th, so velocity R3 passes (at least 90%). The anomaly model passes as well. I expected the load per partition to become uneven. R4 passes if my computer keeps up.

### Observation

Routine: `Ctrl+C` on agents and card processor, `python connector.py fraud-connector.json`, `python card_processor.py --tps 500`, `python agents.py`, scoreboard after about 100 s (card processor and agents still running).

#### 1. Key of an event (Kafka UI, expanded row)

```
"schema": {"type": "struct", "fields": [{"type": "int32", "optional": true, "field": "user_id"}], "optional": false, "name": "fraud.public.transactions.Key"},
"payload": {"user_id": 159607}
```

The key is now only the cardholder (`user_id`) instead of the payment id.

#### 2. Scoreboard

```
Run 1795003305: started 13:01:45, 50,106 transactions in 100 s (501/s), 9 frauds so far (still running)

Consumer groups on the CDC topic(s)   [R2]
  group                    topic                              partitions  members       lag
  anomaly-agent            fraud.public.transactions                   6        1     1,818
  velocity-agent           fraud.public.transactions                   6        3     1,862

Agent instances (telemetry of this run)   [R3, R4]
  instance         pid model      transactions  cardholders  out-of-order  lag (s)      seen
  ano-1          10960 anomaly          48,175       38,204             0      3.9       7s ago
  vel-1          23156 velocity         14,701       11,924             9      4.3       7s ago
  vel-2          22752 velocity         15,172       12,291             8      3.9       7s ago
  vel-3           8240 velocity         14,517       11,751            11      4.2       7s ago

Alerts on 'fraud-alerts' (committed, this run)

  velocity model (card testing)
    R3  frauds caught          4/4  (100%)   PASS  (>= 90 %)
    R3  false alerts           0   PASS  (at most 10 % of the flagged transactions)
    R4  alert latency p50/p95  3.7 s / 6.6 s   FAIL  (p95 <= 5 s after the commit)
        duplicate alerts       0   (info: alerts case management got twice)

  anomaly model (account takeover)
    R3  frauds caught          4/4  (100%)   PASS  (>= 90 %)
    R3  false alerts           0   PASS  (at most 10 % of the flagged transactions)
    R4  alert latency p50/p95  3.5 s / 5.3 s   FAIL  (p95 <= 5 s after the commit)
        duplicate alerts       0   (info: alerts case management got twice)
```

#### 3. Kafka UI: partitions of `fraud.public.transactions` (Overview)

| Partition | Messages |
|---|---|
| 0 | 15,493 |
| 1 | 15,521 |
| 2 | 15,440 |
| 3 | 15,252 |
| 4 | 15,213 |
| 5 | 15,556 |
| Total | 92,475 |

The topic has 6 partitions, replication factor 1. The partitions are filled evenly (15,213 to 15,556), so my prediction of an uneven load was wrong. My interpretation: with 100,000 cardholders, hashing the cardholder id spreads the keys evenly over 6 partitions. An uneven load would need few keys or a few very busy cardholders (not verified against `card_processor.py`).

### Comparison of the three runs (my measurements)

| | Run 1: defaults | Run 2: 6 partitions | Run 3: key = cardholder |
|---|---|---|---|
| Partitions | 1 | 6 | 6 |
| Key | `{"id": ...}` | `{"id": ...}` | `{"user_id": ...}` |
| Velocity instances working | 1 of 3 (2 idle) | 3 of 3 | 3 of 3 |
| Velocity R3 (frauds caught) | 1/7 (14%) | 0/22 (0%) | **4/4 (100%)** |
| Anomaly R3 | 3/8 (38%) | 8/22 (36%) | **4/4 (100%)** |
| Agent lag | 25 s → 109 s, growing | 15 s → 166 s, growing | about 4 s, stable |
| R4 (p95 ≤ 5 s) | FAIL (108.9 s) | FAIL (backlog) | FAIL narrowly (6.6 s / 5.3 s) |
| Card processor rate | about 667/s | about 950/s | 500/s (`--tps 500`) |

Runs 1 and 2 were measured while the agents were far behind, so their R3 values are partly affected by the backlog. Run 3 is the only run where the agents kept up.

### Explanation

The key of every event is now the cardholder, `{"user_id": ...}`, instead of the payment id. Kafka's partition is `hash(key) mod 6`, so all payments of one cardholder go to the same partition and are read by one instance, in commit order. The 12 test payments of a stolen card therefore meet in one velocity instance, which counts 12 (at least 8) in its 60 s window and flags the 8th. Velocity R3 went from 0 of 22 in Run 2 to **4 of 4 (100%)**, and the anomaly model also caught 4 of 4, with no false alerts and no duplicates.

Using the cardholder as key did not make the partitions uneven (about 15,200 to 15,600 messages each), because there are many cardholders.

R4 narrowly fails in my run (p95 6.6 s for velocity, 5.3 s for anomaly, p50 about 3.5 s). The lag of the agents is stable at about 4 s and not growing, so the pipeline keeps up, but it has no headroom on my laptop, where PostgreSQL, Kafka, Connect, the Kafka UI, the card processor and four agents all run together. I did not verify the exact cause. The assignment suggests a fourth velocity instance (`python agents.py --velocity 4`) or a lower rate as fixes. I used `--tps 500`, so my R4 result is for half of the nominal 1,000 payments/s.

**The rule:** the key is the entity whose state the model keeps (here the cardholder), not the primary key of the table. Partitions cap parallelism (Run 1), and the key decides which payments meet in the same instance (Runs 2 and 3).

### Beyond this lab (for reading)

A production pipeline must also handle crashes (a crashed instance loses its in-memory per-cardholder counts, so a fraud in progress can be missed), duplicates (a restarted agent re-reads payments after its last committed offset, so some alerts can arrive twice and can be dropped by `alert_id`), and catch-up after an outage (size instances and partitions with headroom, not just for the normal rate).

Both defaults that Debezium chose for me were wrong for a stateful model that has to scale: the number of partitions (1) and the key (the primary key). Both are fine for Debezium's default job of copying a table. Note how they failed: nothing crashed and no error message appeared. Without the scoreboard, the fraud model would simply have caught fewer frauds.

A threshold rule (flag the 8th payment within 60 s) cannot tell a thief from an honest cardholder who pays 8 times in a minute. That is the price of any threshold, and R3 allows a few false alerts. I saw none in my runs.